# 🏠 House Prices - Modelování

Tento notebook obsahuje **trénování** různých ML modelů pro predikci cen nemovitostí.

## 📋 Struktura:

1. **EXPERIMENT 1:** Baseline Linear Models (Ridge, Lasso)
2. **EXPERIMENT 2:** Tree-based Models (Random Forest, XGBoost, LightGBM)
3. **EXPERIMENT 3:** Hyperparameter Tuning (GridSearchCV)
4. **Uložení modelů** pro evaluaci v notebooku Evaluation.ipynb

---

## 🎯 Experimentální přístup:

Každý experiment obsahuje:
- **Konfiguraci** s dokumentací rozhodnutí (PROČ jsme použili tyto parametry)
- **Trénování modelů** na train data
- **Uložení modelů** do `models/` pro pozdější evaluaci

---

## ⚠️ DŮLEŽITÉ:

- Data se načítají z `data/processed_data.pkl` (ne z paměti!)
- Modely se ukládají do `models/` pro evaluaci v notebooku Evaluation.ipynb
- **Evaluace a porovnání** probíhá v notebooku Evaluation.ipynb
- Každé rozhodnutí je **zdokumentované** pro obhajitelnost na zkoušce


In [3]:
# 📦 Import knihoven
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ML modely
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 📍 Nastavení cest
PROJECT_DIR = Path('..')
DATA_DIR = PROJECT_DIR / 'data'
MODELS_DIR = PROJECT_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# Nastavení pro reprodukovatelnost
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Všechny knihovny načteny!")


✅ Všechny knihovny načteny!


## 1. Načtení preprocessovaných dat


In [4]:
# Načtení preprocessovaných dat
with open(DATA_DIR / 'processed_data.pkl', 'rb') as f:
    processed_data = pickle.load(f)
    X_train = processed_data['X_train']
    X_test = processed_data['X_test']
    y_train = processed_data['y_train']
    test_ids = processed_data['test_ids']

print("=" * 80)
print("📊 NAČTENÁ DATA")
print("=" * 80)
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_train: {y_train.shape} (log SalePrice)")
print(f"   Features: {X_train.shape[1]}")
print("\n✅ Data připravena pro modelování!")


📊 NAČTENÁ DATA
   X_train: (1458, 248)
   X_test: (1459, 248)
   y_train: (1458,) (log SalePrice)
   Features: 248

✅ Data připravena pro modelování!


In [5]:
# Vyplnění NaN hodnot - DŮLEŽITÉ pro linear modely!
print("=" * 80)
print("🔍 VYPLNĚNÍ NaN HODNOT")
print("=" * 80)

# Zkontrolujeme NaN
train_nan = X_train.isnull().sum().sum()
test_nan = X_test.isnull().sum().sum()
print(f"   X_train NaN: {train_nan}")
print(f"   X_test NaN: {test_nan}")

# Vyplníme všechny NaN hodnoty
# Pro numerické: median z train
# Pro ostatní: 0
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if X_train[col].isnull().any() or X_test[col].isnull().any():
        fill_val = X_train[col].median()
        if pd.isna(fill_val):
            fill_val = 0
        X_train[col] = X_train[col].fillna(fill_val)
        X_test[col] = X_test[col].fillna(fill_val)

# Pro ostatní sloupce: 0
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# Finální kontrola
train_nan_final = X_train.isnull().sum().sum()
test_nan_final = X_test.isnull().sum().sum()
print(f"\n   ✅ Po vyplnění:")
print(f"      X_train NaN: {train_nan_final}")
print(f"      X_test NaN: {test_nan_final}")
print("\n✅ Data připravena pro modelování (bez NaN)!")


🔍 VYPLNĚNÍ NaN HODNOT
   X_train NaN: 118
   X_test NaN: 122

   ✅ Po vyplnění:
      X_train NaN: 0
      X_test NaN: 0

✅ Data připravena pro modelování (bez NaN)!


## 2. EXPERIMENT 1: Baseline Linear Models

**Cíl:** Otestovat základní linear modely s regularizací jako baseline.

**Modely:**
- **Ridge Regression** - L2 regularizace (snižuje overfitting)
- **Lasso Regression** - L1 regularizace (feature selection)


In [6]:
# Konfigurace experimentu
EXP1_CONFIG = {
    'experiment_id': 'EXP_001',
    'name': 'Baseline Linear Models',
    'cv_folds': 5,
    'cv_reason': '5-fold CV je standardní praxe, poskytuje dobrý kompromis mezi bias a variance',
    'models': {
        'Ridge': {
            'alpha': 10.0,
            'reason': 'Alpha=10.0 je rozumná výchozí hodnota pro L2 regularizaci'
        },
        'Lasso': {
            'alpha': 0.001,
            'reason': 'Alpha=0.001 je menší než Ridge, protože Lasso má silnější regularizaci'
        }
    }
}

print("=" * 80)
print(f"🔬 {EXP1_CONFIG['name']}")
print("=" * 80)
print(f"   CV folds: {EXP1_CONFIG['cv_folds']}")
print(f"   Důvod: {EXP1_CONFIG['cv_reason']}")
print(f"   Modely: {', '.join(EXP1_CONFIG['models'].keys())}")


🔬 Baseline Linear Models
   CV folds: 5
   Důvod: 5-fold CV je standardní praxe, poskytuje dobrý kompromis mezi bias a variance
   Modely: Ridge, Lasso


In [7]:
# Trénování modelů
exp1_models = {}
exp1_results = {}

# Ridge Regression
print("\n📊 Trénování Ridge Regression...")
ridge = Ridge(alpha=EXP1_CONFIG['models']['Ridge']['alpha'], random_state=RANDOM_STATE)
ridge.fit(X_train, y_train)
exp1_models['Ridge'] = ridge

# Lasso Regression
print("📊 Trénování Lasso Regression...")
lasso = Lasso(alpha=EXP1_CONFIG['models']['Lasso']['alpha'], random_state=RANDOM_STATE)
lasso.fit(X_train, y_train)
exp1_models['Lasso'] = lasso

print("\n✅ Všechny modely natrénovány!")



📊 Trénování Ridge Regression...
📊 Trénování Lasso Regression...

✅ Všechny modely natrénovány!


In [8]:
# Uložení modelů
print("=" * 80)
print("💾 UKLÁDÁNÍ MODELŮ")
print("=" * 80)

for name, model in exp1_models.items():
    model_path = MODELS_DIR / f'exp1_{name.lower()}.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"   ✅ {name} uložen: {model_path}")

# Uložení konfigurace
config_path = MODELS_DIR / 'exp1_config.pkl'
with open(config_path, 'wb') as f:
    pickle.dump(EXP1_CONFIG, f)
print(f"\n✅ Konfigurace uložena: {config_path}")


💾 UKLÁDÁNÍ MODELŮ
   ✅ Ridge uložen: ..\models\exp1_ridge.pkl
   ✅ Lasso uložen: ..\models\exp1_lasso.pkl

✅ Konfigurace uložena: ..\models\exp1_config.pkl


### 📝 Závěr EXPERIMENT 1:

**Natrénované modely:**
- Ridge Regression
- Lasso Regression

**Modely uloženy** pro evaluaci v notebooku Evaluation.ipynb


## 3. EXPERIMENT 2: Tree-based Models

**Cíl:** Otestovat pokročilé tree-based modely, které obvykle dosahují lepších výsledků než linear modely.

**Modely:**
- **Random Forest** - Ensemble bagging metoda
- **XGBoost** - Gradient boosting, často nejlepší výsledky
- **LightGBM** - Rychlý gradient boosting


In [9]:
# Konfigurace experimentu
EXP2_CONFIG = {
    'experiment_id': 'EXP_002',
    'name': 'Tree-based Models',
    'cv_folds': 5,
    'cv_reason': 'Stejná CV strategie jako Experiment 1 pro objektivní porovnání',
    'models': {
        'RandomForest': {
            'n_estimators': 100,
            'max_depth': 15,
            'reason': '100 trees je kompromis mezi rychlostí a přesností. Depth 15 zabraňuje overfittingu.'
        },
        'XGBoost': {
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'max_depth': 5,
            'reason': 'Learning rate 0.05 je konzervativní, zabraňuje overfittingu. Depth 5 je standardní.'
        },
        'LightGBM': {
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'max_depth': 5,
            'reason': 'Stejné parametry jako XGBoost pro spravedlivé porovnání.'
        }
    }
}

print("=" * 80)
print(f"🔬 {EXP2_CONFIG['name']}")
print("=" * 80)
print(f"   CV folds: {EXP2_CONFIG['cv_folds']}")
print(f"   Důvod: {EXP2_CONFIG['cv_reason']}")
print(f"   Modely: {', '.join(EXP2_CONFIG['models'].keys())}")


🔬 Tree-based Models
   CV folds: 5
   Důvod: Stejná CV strategie jako Experiment 1 pro objektivní porovnání
   Modely: RandomForest, XGBoost, LightGBM


In [10]:
# Trénování modelů
exp2_models = {}
exp2_results = {}

# Random Forest
print("\n📊 Trénování Random Forest...")
rf = RandomForestRegressor(
    n_estimators=EXP2_CONFIG['models']['RandomForest']['n_estimators'],
    max_depth=EXP2_CONFIG['models']['RandomForest']['max_depth'],
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)
exp2_models['RandomForest'] = rf

# XGBoost
print("📊 Trénování XGBoost...")
xgb = XGBRegressor(
    n_estimators=EXP2_CONFIG['models']['XGBoost']['n_estimators'],
    learning_rate=EXP2_CONFIG['models']['XGBoost']['learning_rate'],
    max_depth=EXP2_CONFIG['models']['XGBoost']['max_depth'],
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
exp2_models['XGBoost'] = xgb

# LightGBM
print("📊 Trénování LightGBM...")
lgbm = LGBMRegressor(
    n_estimators=EXP2_CONFIG['models']['LightGBM']['n_estimators'],
    learning_rate=EXP2_CONFIG['models']['LightGBM']['learning_rate'],
    max_depth=EXP2_CONFIG['models']['LightGBM']['max_depth'],
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(X_train, y_train)
exp2_models['LightGBM'] = lgbm

print("\n✅ Všechny modely natrénovány!")



📊 Trénování Random Forest...
📊 Trénování XGBoost...
📊 Trénování LightGBM...

✅ Všechny modely natrénovány!


In [11]:
# Uložení modelů
print("=" * 80)
print("💾 UKLÁDÁNÍ MODELŮ")
print("=" * 80)

for name, model in exp2_models.items():
    model_path = MODELS_DIR / f'exp2_{name.lower()}.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"   ✅ {name} uložen: {model_path}")

# Uložení konfigurace
config_path = MODELS_DIR / 'exp2_config.pkl'
with open(config_path, 'wb') as f:
    pickle.dump(EXP2_CONFIG, f)
print(f"\n✅ Konfigurace uložena: {config_path}")

# Uložení informace o nejlepším modelu (pro EXP3)
best_exp2 = 'XGBoost'  # Výchozí, bude upraveno v evaluaci
print(f"\n💡 Pro Experiment 3 použijeme nejlepší model z evaluace")


💾 UKLÁDÁNÍ MODELŮ
   ✅ RandomForest uložen: ..\models\exp2_randomforest.pkl
   ✅ XGBoost uložen: ..\models\exp2_xgboost.pkl
   ✅ LightGBM uložen: ..\models\exp2_lightgbm.pkl

✅ Konfigurace uložena: ..\models\exp2_config.pkl

💡 Pro Experiment 3 použijeme nejlepší model z evaluace


### 📝 Závěr EXPERIMENT 2:

**Natrénované modely:**
- Random Forest
- XGBoost
- LightGBM

**Modely uloženy** pro evaluaci v notebooku Evaluation.ipynb


## 4. EXPERIMENT 3: Hyperparameter Tuning

**Cíl:** Optimalizovat hyperparametry nejlepšího modelu z Experiment 2 pomocí GridSearchCV.

**Metoda:** GridSearchCV s cross-validation (stejná CV strategie jako předtím)


In [12]:
# Konfigurace experimentu
# POZNÁMKA: Nejlepší model z EXP2 bude určen v Evaluation.ipynb

EXP3_CONFIG = {
    'experiment_id': 'EXP_003',
    'name': 'Hyperparameter Tuning',
    'base_model': 'LightGBM',  
    'cv_folds': 5,
    'cv_reason': 'Stejná CV strategie pro konzistentní porovnání',
    'param_grid': {
        'XGBoost': {
            'n_estimators': [500, 1000, 1500],
            'learning_rate': [0.03, 0.05, 0.07],
            'max_depth': [4, 5, 6],
            'reason': 'Testujeme rozumný rozsah hodnot kolem výchozích parametrů'
        },
        'LightGBM': {
            'n_estimators': [500, 1000, 1500],
            'learning_rate': [0.03, 0.05, 0.07],
            'max_depth': [4, 5, 6],
            'reason': 'Stejný grid jako XGBoost pro spravedlivé porovnání'
        },
        'RandomForest': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 15, 20],
            'reason': 'RF je pomalejší, takže menší grid'
        }
    }
}

print("=" * 80)
print(f"🔬 {EXP3_CONFIG['name']}")
print("=" * 80)
print(f"   Base model: {EXP3_CONFIG['base_model']} ")
print(f"   CV folds: {EXP3_CONFIG['cv_folds']}")
print(f"   Důvod: {EXP3_CONFIG['cv_reason']}")


🔬 Hyperparameter Tuning
   Base model: LightGBM 
   CV folds: 5
   Důvod: Stejná CV strategie pro konzistentní porovnání


In [13]:
# GridSearchCV pro nejlepší model z EXP2

base_model_name = EXP3_CONFIG['base_model']
print(f"\n🔍 GridSearchCV pro {base_model_name}...")
print("💡 POZNÁMKA: V produkci použij nejlepší model z evaluace EXP2!")

if base_model_name == 'XGBoost':
    base_model = XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    param_grid = EXP3_CONFIG['param_grid']['XGBoost'].copy()
    del param_grid['reason']
    
elif base_model_name == 'LightGBM':
    base_model = LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    param_grid = EXP3_CONFIG['param_grid']['LightGBM'].copy()
    del param_grid['reason']
    
else:  # RandomForest
    base_model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    param_grid = EXP3_CONFIG['param_grid']['RandomForest'].copy()
    del param_grid['reason']

# GridSearchCV
kfold = KFold(n_splits=EXP3_CONFIG['cv_folds'], shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=kfold,
    n_jobs=-1,
    verbose=1
)

print("⏳ Trénování (může to chvíli trvat)...")
grid_search.fit(X_train, y_train)

print(f"\n✅ GridSearchCV dokončen!")
print(f"   Best CV RMSE: {-grid_search.best_score_:.4f}")
print(f"   Best Parameters: {grid_search.best_params_}")

# Uložení nejlepšího modelu
exp3_best_model = grid_search.best_estimator_
model_path = MODELS_DIR / f'exp3_{base_model_name.lower()}_tuned.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(exp3_best_model, f)
print(f"\n✅ Model uložen: {model_path}")

# Uložení konfigurace a výsledků
exp3_info = {
    'best_params': grid_search.best_params_,
    'best_cv_rmse': -grid_search.best_score_,
    'base_model': base_model_name
}
config_path = MODELS_DIR / 'exp3_config.pkl'
with open(config_path, 'wb') as f:
    pickle.dump({**EXP3_CONFIG, **exp3_info}, f)
print(f"✅ Konfigurace uložena: {config_path}")



🔍 GridSearchCV pro LightGBM...
💡 POZNÁMKA: V produkci použij nejlepší model z evaluace EXP2!
⏳ Trénování (může to chvíli trvat)...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

✅ GridSearchCV dokončen!
   Best CV RMSE: 0.1208
   Best Parameters: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 500}

✅ Model uložen: ..\models\exp3_lightgbm_tuned.pkl
✅ Konfigurace uložena: ..\models\exp3_config.pkl


### 📝 Závěr EXPERIMENT 3:

**Natrénovaný model:**
- Tuned model (GridSearchCV)

**Model uložen** pro evaluaci v notebooku Evaluation.ipynb

---

## ✅ Shrnutí trénování:

Všechny modely byly natrénovány a uloženy do `models/`:
- **EXP1:** Ridge, Lasso
- **EXP2:** RandomForest, XGBoost, LightGBM
- **EXP3:** Tuned model (GridSearchCV)

**Další krok:** Spusť notebooku Evaluation.ipynb pro evaluaci a porovnání všech modelů!
